# Quickstart with Microsoft Agent Framework

## Part 1: Agents

In [1]:
// Requirements and Client Configuration
#r "nuget: Microsoft.Agents.AI.OpenAI, *-*"
using System.ComponentModel;
using System.Threading;
using System.Text.Json;
using System.Text;
using Microsoft.Agents.AI;
using OpenAI;
using Microsoft.Extensions.AI;
using System.Net.Http;

var key = await Microsoft.DotNet.Interactive.Kernel.GetPasswordAsync("Enter your OpenAI API Key") ?? throw new InvalidOperationException("OPENAI_API_KEY is not set.");

OpenAIClient client = new(key.GetClearTextPassword());
OpenAI.Chat.ChatClient chatClient = client.GetChatClient("gpt-4o-mini");

Installed Packages Microsoft.Agents.AI.OpenAI, 1.0.0-preview.251028.1

In [2]:
// Demo 1: Simple Agent

AIAgent agent1 = chatClient.CreateAIAgent(instructions: "You are good at telling jokes.");
Console.WriteLine("Joke Agent: " + await agent1.RunAsync("Tell me a funny joke."));

Joke Agent: Why did the scarecrow win an award? 

Because he was outstanding in his field!


In [3]:
// Demo 2: Specialized Agent

AIAgent agent2 = chatClient.CreateAIAgent(
    instructions: "You are a math expert that always answers short and concise with no question or explanation.",
    name: "MathAgent",
    description: "An agent that answers math questions.");

string question = "log(100) + 5 * 3";
Console.WriteLine("Math Agent: " + await agent2.RunAsync(question));

Math Agent: Log(100) = 2, so the expression evaluates to 2 + 15 = 17.


In [ ]:
// Demo 3: Multi-Agent Conversation with Context Sharing

AIAgent agent3 = chatClient.CreateAIAgent(instructions: "You are good at telling stories.", name: "StoryAgent", description: "An agent that tells stories.");

AgentThread thread = agent3.GetNewThread();
await foreach (var update in agent3.RunStreamingAsync("Tell me a very short story about a pirate.", thread))
{
    Console.Write(update);
}


Console.WriteLine("Story with Emojis:");

// Create an emoji agent to enhance the story
AIAgent emojiAgent = chatClient.CreateAIAgent(
    instructions: "You add relevant emojis to text. Keep the original text intact and add emojis next to appropriate words.", 
    name: "EmojiAgent", 
    description: "An agent that adds emojis to text.");


// Use the same thread for multi-turn conversation
await foreach (var update in emojiAgent.RunStreamingAsync("Add emojis to the story above.", thread))
{
    Console.Write(update);
}

In [14]:
// Demo 4: Agent with Tool Use

[Description("Get the weather for a given location.")]
static string GetWeather([Description("The location to get the weather for.")] string location)
    => $"The weather in {location} is cloudy with a high of 15°C.";

// create a new agent with a tool GetWeather
AIAgent weatherAgent = chatClient.CreateAIAgent(
        instructions: "You answer questions about the weather. answer short and concise with no question or explanation.",
        name: "WeatherAgent",
        description: "An agent that answers questions about the weather.",
        tools: [AIFunctionFactory.Create(GetWeather)]);


Console.WriteLine("Weather Agent: " + await weatherAgent.RunAsync("What's the weather in Seattle?"));

Weather Agent: The weather in Seattle is cloudy with a high of 15°C.


In [15]:
// Demo 5: Image Generation

var imageClient = client.GetImageClient("gpt-image-1-mini");

var prompt = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Describe the image you want to generate:") ?? "Generic scenic landscape";
var imageResponse = await imageClient.GenerateImageAsync(prompt, new OpenAI.Images.ImageGenerationOptions() { Size = OpenAI.Images.GeneratedImageSize.W1024xH1024 });

var imageBytes = imageResponse.Value.ImageBytes;

// Show the image in the notebook
var base64 = Convert.ToBase64String(imageBytes.ToArray());
var html = $"<img src=\"data:image/png;base64,{base64}\" width=\"400\" height=\"400\" />";
html.DisplayAs("text/html");

Error: Input request cancelled

Error: System.Exception: Input request cancelled
   at Microsoft.DotNet.Interactive.Kernel.GetInputAsync(String prompt, Boolean isPassword, String typeHint) in D:\a\_work\1\s\src\Microsoft.DotNet.Interactive\Kernel.Static.cs:line 93
   at Microsoft.DotNet.Interactive.Kernel.GetInputAsync(String prompt, String typeHint) in D:\a\_work\1\s\src\Microsoft.DotNet.Interactive\Kernel.Static.cs:line 46
   at Submission#15.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [ ]:
// Demo 6: Agent with Memory

#nullable enable

internal sealed class UserInfoMemory : AIContextProvider
{
    private readonly IChatClient _chatClient;
    public UserInfoMemory(IChatClient chatClient, UserInfo? userInfo = null)
    {
        this._chatClient = chatClient;
        this.UserInfo = userInfo ?? new UserInfo();
    }

    public UserInfoMemory(IChatClient chatClient, JsonElement serializedState, JsonSerializerOptions? jsonSerializerOptions = null)
    {
        this._chatClient = chatClient;
        this.UserInfo = serializedState.ValueKind == JsonValueKind.Object ?
            serializedState.Deserialize<UserInfo>(jsonSerializerOptions)! :
            new UserInfo();
    }

    public UserInfo UserInfo { get; set; }

    public override async ValueTask InvokedAsync(
        InvokedContext context,
        CancellationToken cancellationToken = default)
    {
        if ((this.UserInfo.UserName is null || this.UserInfo.UserAge is null) && context.RequestMessages.Any(x => x.Role == ChatRole.User))
        {
            var result = await this._chatClient.GetResponseAsync<UserInfo>(
                context.RequestMessages,
                new ChatOptions()
                {
                    Instructions = "Extract the user's name and age from the message if present. If not present return nulls."
                },
                cancellationToken: cancellationToken);
            this.UserInfo.UserName ??= result.Result.UserName;
            this.UserInfo.UserAge ??= result.Result.UserAge;
        }
    }

    public override ValueTask<AIContext> InvokingAsync(
        InvokingContext context,
        CancellationToken cancellationToken = default)
    {
        StringBuilder instructions = new();
        instructions
            .AppendLine(
                this.UserInfo.UserName is null ?
                    "Ask the user for their name and politely decline to answer any questions until they provide it." :
                    $"The user's name is {this.UserInfo.UserName}.")
            .AppendLine(
                this.UserInfo.UserAge is null ?
                    "Ask the user for their age and politely decline to answer any questions until they provide it." :
                    $"The user's age is {this.UserInfo.UserAge}.");
        return new ValueTask<AIContext>(new AIContext
        {
            Instructions = instructions.ToString()
        });
    }

    public override JsonElement Serialize(JsonSerializerOptions? jsonSerializerOptions = null)
    {
        return JsonSerializer.SerializeToElement(this.UserInfo, jsonSerializerOptions);
    }
}

internal sealed class UserInfo
{
    public string? UserName { get; set; }
    public int? UserAge { get; set; }
}

AIAgent memoryAgent = chatClient.CreateAIAgent(new ChatClientAgentOptions()
{
    Instructions = "You are a friendly assistant. Always address the user by their name.",
    AIContextProviderFactory = ctx => new UserInfoMemory(
        chatClient.AsIChatClient(),
        ctx.SerializedState,
        ctx.JsonSerializerOptions)
});
// Create a new thread for the conversation.
AgentThread memoryThread = memoryAgent.GetNewThread();

Console.WriteLine(await memoryAgent.RunAsync("Hello, what is the square root of 9?", memoryThread));
Console.WriteLine(await memoryAgent.RunAsync("My name is Sepehr", memoryThread));
Console.WriteLine(await memoryAgent.RunAsync("I am 43 years old", memoryThread));

// Access the memory component via the thread's GetService method.
var userInfo = memoryThread.GetService<UserInfoMemory>()?.UserInfo;
Console.WriteLine($"MEMORY - User Name: {userInfo?.UserName}");
Console.WriteLine($"MEMORY - User Age: {userInfo?.UserAge}");


In [ ]:
// Demo 7: Financial Advisor Multi-Agent System - Combining Tool Use, Specialized Agents, and Context Sharing 
var alphaVantageKey = await Microsoft.DotNet.Interactive.Kernel.GetPasswordAsync("Enter your Alpha Vantage API Key") ?? throw new InvalidOperationException("ALPHA_VANTAGE_API_KEY is not set.");

static double CalculateRSI(List<double> prices, int period = 14)
{
    if (prices.Count < period + 1) return 50.0;

    var (gains, losses) = (0.0, 0.0);
    for (int i = 0; i < period; i++)
    {
        var change = prices[i] - prices[i + 1];
        if (change > 0) gains += change; else losses -= change;
    }

    var (avgGain, avgLoss) = (gains / period, losses / period);
    return avgLoss == 0 ? 100.0 : 100 - (100 / (1 + avgGain / avgLoss));
}

static (double upper, double lower) CalculateBollingerBands(List<double> prices, int period = 20)
{
    if (prices.Count < period) return (0, 0);
    
    var recentPrices = prices.Take(period);
    var ma = recentPrices.Average();
    var stdDev = Math.Sqrt(recentPrices.Select(p => Math.Pow(p - ma, 2)).Average());
    
    return (ma + 2 * stdDev, ma - 2 * stdDev);
}

[Description("Fetches fundamental and technical data for a stock ticker from Alpha Vantage.")]
async Task<string> FetchStockData([Description("Stock ticker symbol (e.g., AAPL, MSFT)")] string ticker)
{
    try
    {
        using var httpClient = new HttpClient();
        var baseUrl = "https://www.alphavantage.co/query";
        
        // Fetch time series data
        var timeSeriesUrl = $"{baseUrl}?function=TIME_SERIES_DAILY&symbol={ticker}&outputsize=full&apikey={alphaVantageKey.GetClearTextPassword()}";
        var timeSeriesJson = JsonDocument.Parse(await httpClient.GetStringAsync(timeSeriesUrl));
        
        if (timeSeriesJson.RootElement.TryGetProperty("Error Message", out _))
            return $"Error: Invalid ticker '{ticker}'";
        if (timeSeriesJson.RootElement.TryGetProperty("Note", out _))
            return "Error: API rate limit reached. Try again in a minute.";
        
        // Parse prices
        var closePrices = timeSeriesJson.RootElement
            .GetProperty("Time Series (Daily)")
            .EnumerateObject()
            .OrderByDescending(d => d.Name)
            .Take(250)
            .Select(d => double.Parse(d.Value.GetProperty("4. close").GetString() ?? "0"))
            .ToList();
        
        if (closePrices.Count == 0) return $"Error: No data for {ticker}";
        
        // Calculate indicators
        var (currentPrice, ma50, ma200) = (
            closePrices[0],
            closePrices.Take(Math.Min(50, closePrices.Count)).Average(),
            closePrices.Take(Math.Min(200, closePrices.Count)).Average()
        );
        var rsi = CalculateRSI(closePrices);
        var (upper, lower) = CalculateBollingerBands(closePrices);
        
        // Fetch fundamentals
        var overviewUrl = $"{baseUrl}?function=OVERVIEW&symbol={ticker}&apikey={alphaVantageKey.GetClearTextPassword()}";
        var overviewJson = JsonDocument.Parse(await httpClient.GetStringAsync(overviewUrl));
        
        var ParseDouble = (string key) => 
            overviewJson.RootElement.TryGetProperty(key, out var el) && 
            double.TryParse(el.GetString(), out var val) ? val : 0.0;
        
        var (peRatio, eps, marketCap) = (ParseDouble("PERatio"), ParseDouble("EPS"), ParseDouble("MarketCapitalization"));

        return $"""
            Stock Data for {ticker.ToUpper()}:
            - Current Price: ${currentPrice:F2}
            - Market Cap: ${marketCap / 1_000_000_000:F2}B
            - P/E Ratio: {peRatio:F2}
            - EPS: ${eps:F2}
            - 50-day MA: ${ma50:F2}
            - 200-day MA: ${ma200:F2}
            - RSI (14-day): {rsi:F2}
            - Bollinger Upper: ${upper:F2}
            - Bollinger Lower: ${lower:F2}
            """;
    }
    catch (Exception ex)
    {
        return $"Error: {ex.Message}";
    }
}

Console.WriteLine("🚀 Financial Advisor Multi-Agent System");
var ticker = (await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Enter stock ticker (e.g., AAPL, MSFT, GOOGL)")).Trim().ToUpper();

// Create specialized agents
var dataAgent = chatClient.CreateAIAgent(
    instructions: "Fetch stock data using the provided tool and return all metrics as a summary.",
    name: "DataFetcher",
    description: "Fetches fundamental and technical data.",
    tools: [AIFunctionFactory.Create(FetchStockData)]);

var fundamentalAgent = chatClient.CreateAIAgent(
    instructions: "Analyze P/E ratio, EPS, and market cap. Determine if the stock is undervalued, fairly valued, or overvalued. Be concise.",
    name: "FundamentalAnalyst",
    description: "Analyzes fundamentals.");

var technicalAgent = chatClient.CreateAIAgent(
    instructions: "Evaluate moving averages, RSI, and Bollinger Bands to determine trend and momentum. Be concise.",
    name: "TechnicalAnalyst",
    description: "Analyzes technical indicators.");

var recommendationAgent = chatClient.CreateAIAgent(
    instructions: "Provide final recommendation: 'Strong Buy', 'Hold', or 'Sell' based on both analyses. Explain briefly.",
    name: "RecommendationAgent",
    description: "Provides buy/hold/sell recommendation.");

// Run multi-agent analysis
Console.WriteLine($"\n📊 Analyzing {ticker}...\n💬 Agent Conversations:\n");

// Step 1: Fetch data
Console.WriteLine("[Data Fetcher]");
var dataResponse = await dataAgent.RunAsync($"Fetch and summarize data for {ticker}");
Console.WriteLine($"{dataResponse}\n");

// Step 2: Run fundamental and technical analysis in parallel
Console.WriteLine("[Running Fundamental & Technical Analysis in Parallel...]\n");

var (fundamentalThread, technicalThread) = (fundamentalAgent.GetNewThread(), technicalAgent.GetNewThread());
var analysisPrompt = $"Analyze this data:\n{dataResponse}";

var fundamentalTask = fundamentalAgent.RunAsync(analysisPrompt, fundamentalThread);
var technicalTask = technicalAgent.RunAsync(analysisPrompt, technicalThread);
Task.WaitAll(fundamentalTask, technicalTask);

var (fundamentalResult, technicalResult) = (fundamentalTask.Result, technicalTask.Result);

Console.WriteLine($"[Fundamental Analyst]\n{fundamentalResult}\n");
Console.WriteLine($"[Technical Analyst]\n{technicalResult}\n");

// Step 3: Synthesize recommendation
Console.WriteLine("[Recommendation Agent]");
var recommendation = recommendationAgent.RunAsync($"""
    Provide final recommendation based on:
    
    Fundamental: {fundamentalResult}
    Technical: {technicalResult}
    """).Result;
Console.WriteLine($"{recommendation}\n\n✅ Analysis Complete!");


# Part 2: Workflows

In [26]:
#r "nuget: Microsoft.Agents.AI.Workflows, *-*"
using System;
using System.Threading.Tasks;
using Microsoft.Agents.AI;
using Microsoft.Agents.AI.Workflows;
using Microsoft.Agents.AI.Workflows.Reflection;
using Microsoft.Extensions.AI;

Installed Packages Microsoft.Agents.AI.Workflows, 1.0.0-preview.251028.1

In [19]:
internal sealed class UppercaseExecutor() : Executor<string, string>("UppercaseExecutor")
{
    public override ValueTask<string> HandleAsync(string message, IWorkflowContext context, CancellationToken cancellationToken = default) =>
        ValueTask.FromResult(message.ToUpperInvariant()); 
}

internal sealed class ReverseTextExecutor() : Executor<string, string>("ReverseTextExecutor")
{
    public override ValueTask<string> HandleAsync(string message, IWorkflowContext context, CancellationToken cancellationToken = default) =>
        ValueTask.FromResult(string.Concat(message.Reverse()));
}

UppercaseExecutor uppercase = new();
ReverseTextExecutor reverse = new();

WorkflowBuilder builder = new(uppercase);
builder.AddEdge(uppercase, reverse).WithOutputFrom(reverse);
var workflow = builder.Build();

// Streaming Option
//await using StreamingRun run = await InProcessExecution.StreamAsync(workflow, "Hello, World!");
var run = await InProcessExecution.RunAsync(workflow, "Hello, World!");
//await foreach (WorkflowEvent evt in run.WatchStreamAsync())
foreach (WorkflowEvent evt in run.NewEvents)
{
    if (evt is ExecutorCompletedEvent executorComplete)
    {
        Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
    }
}


UppercaseExecutor: HELLO, WORLD!
ReverseTextExecutor: !DLROW ,OLLEH


In [23]:
private static ChatClientAgent GetTranslationAgent(string targetLanguage, OpenAI.Chat.ChatClient chatClient) =>
    new(chatClient.AsIChatClient(), $"You are a translation assistant that translates the provided text to {targetLanguage}.");


{
        // Create agents
        AIAgent frenchAgent = GetTranslationAgent("French", chatClient);
        AIAgent spanishAgent = GetTranslationAgent("Spanish", chatClient);
        AIAgent englishAgent = GetTranslationAgent("English", chatClient);

        // Build the workflow by adding executors and connecting them
        var workflow = new WorkflowBuilder(frenchAgent)
            .AddEdge(frenchAgent, spanishAgent)
            .AddEdge(spanishAgent, englishAgent)
            .Build();

        // Execute the workflow
        await using StreamingRun run = await InProcessExecution.StreamAsync(workflow, new ChatMessage(ChatRole.User, "Hello World!"));


        await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
        await foreach (WorkflowEvent evt in run.WatchStreamAsync())
        {
            if (evt is AgentRunUpdateEvent executorComplete)
            {
                Console.WriteLine($"{executorComplete.ExecutorId}: {executorComplete.Data}");
            }
        }
}

8c2f95aaef644708a267208e1bf9490e: 
8c2f95aaef644708a267208e1bf9490e: Bonjour
8c2f95aaef644708a267208e1bf9490e:  le
8c2f95aaef644708a267208e1bf9490e:  monde
8c2f95aaef644708a267208e1bf9490e:  !
8c2f95aaef644708a267208e1bf9490e: 
8c2f95aaef644708a267208e1bf9490e: 
908bae8424c74436a5ae1da895d72a8b: 
908bae8424c74436a5ae1da895d72a8b: ¡
908bae8424c74436a5ae1da895d72a8b: Hola
908bae8424c74436a5ae1da895d72a8b: ,
908bae8424c74436a5ae1da895d72a8b:  mundo
908bae8424c74436a5ae1da895d72a8b: !
908bae8424c74436a5ae1da895d72a8b: 
908bae8424c74436a5ae1da895d72a8b: 
cee10673c57a4c108d522c8f0627dacd: 
cee10673c57a4c108d522c8f0627dacd: Hello
cee10673c57a4c108d522c8f0627dacd: ,
cee10673c57a4c108d522c8f0627dacd:  world
cee10673c57a4c108d522c8f0627dacd: !
cee10673c57a4c108d522c8f0627dacd: 
cee10673c57a4c108d522c8f0627dacd: 


In [28]:
using System.Collections.Generic;
using System.Linq;
using System.Threading;
using System.Threading.Tasks;

// Simple round-robin group chat orchestrator for environments lacking the packaged manager.
internal sealed class SimpleRoundRobinGroupChat
{
    private readonly IReadOnlyList<AIAgent> _agents;
    private readonly int _maximumIterationCount;

    public SimpleRoundRobinGroupChat(IReadOnlyList<AIAgent> agents, int maximumIterationCount)
    {
        if (agents is null || agents.Count == 0)
        {
            throw new ArgumentException("At least one agent is required.", nameof(agents));
        }

        foreach (var agent in agents)
        {
            if (agent is null)
            {
                throw new ArgumentException("All agents must be non-null.", nameof(agents));
            }
        }

        if (maximumIterationCount < 1)
        {
            throw new ArgumentOutOfRangeException(nameof(maximumIterationCount));
        }

        this._agents = agents;
        this._maximumIterationCount = maximumIterationCount;
    }

    public async Task<List<ChatMessage>> RunAsync(List<ChatMessage> seedMessages, CancellationToken cancellationToken = default)
    {
        ArgumentNullException.ThrowIfNull(seedMessages);

        List<ChatMessage> history = new(seedMessages);

        for (int iteration = 0; iteration < this._maximumIterationCount; iteration++)
        {
            var agent = this._agents[iteration % this._agents.Count];

            Console.WriteLine();
            Console.WriteLine(agent.DisplayName);

            List<AgentRunResponseUpdate> updates = [];

            await foreach (var update in agent.RunStreamingAsync(history, cancellationToken: cancellationToken))
            {
                updates.Add(update);
                Console.Write(update.Text);

                if (update.Contents.OfType<FunctionCallContent>().FirstOrDefault() is FunctionCallContent call)
                {
                    Console.WriteLine();
                    Console.WriteLine($"  [Calling function '{call.Name}' with arguments: {JsonSerializer.Serialize(call.Arguments)}]");
                }
            }

            var response = updates.ToAgentRunResponse();
            if (response.Messages.Count == 0)
            {
                continue;
            }

            history.AddRange(response.Messages);
        }

        Console.WriteLine();
        return history;
    }
}

In [ ]:
static async Task<List<ChatMessage>> RunWorkflowAsync(Workflow workflow, List<ChatMessage> messages)
{
    string? lastExecutorId = null;

    await using StreamingRun run = await InProcessExecution.StreamAsync(workflow, messages);
    await run.TrySendMessageAsync(new TurnToken(emitEvents: true));
    await foreach (WorkflowEvent evt in run.WatchStreamAsync())
    {
        if (evt is AgentRunUpdateEvent e)
        {
            if (e.ExecutorId != lastExecutorId)
            {
                lastExecutorId = e.ExecutorId;
                Console.WriteLine();
                Console.WriteLine(e.ExecutorId);
            }

            Console.Write(e.Update.Text);
            if (e.Update.Contents.OfType<FunctionCallContent>().FirstOrDefault() is FunctionCallContent call)
            {
                Console.WriteLine();
                Console.WriteLine($"  [Calling function '{call.Name}' with arguments: {JsonSerializer.Serialize(call.Arguments)}]");
            }
        }
        else if (evt is WorkflowOutputEvent output)
        {
            Console.WriteLine();
            return output.As<List<ChatMessage>>()!;
        }
    }

    return [];
}


Console.Write("Choose workflow type ('sequential', 'concurrent', 'handoffs', 'groupchat'): ");
switch (await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("sequential, concurrent, handoffs"))
{
    case "sequential":
        await RunWorkflowAsync(
            AgentWorkflowBuilder.BuildSequential(from lang in (string[])["French", "Spanish", "English"] select GetTranslationAgent(lang, chatClient)),
            [new(ChatRole.User, "Hello, world!")]);
        break;

    case "concurrent":
        await RunWorkflowAsync(
            AgentWorkflowBuilder.BuildConcurrent(from lang in (string[])["French", "Spanish", "English"] select GetTranslationAgent(lang, chatClient)),
            [new(ChatRole.User, "Hello, world!")]);
        break;

    case "handoffs":
        ChatClientAgent historyTutor = new(chatClient.AsIChatClient(),
            "You provide assistance with historical queries. Explain important events and context clearly. Only respond about history.",
            "history_tutor",
            "Specialist agent for historical questions");
        ChatClientAgent mathTutor = new(chatClient.AsIChatClient(),
            "You provide help with math problems. Explain your reasoning at each step and include examples. Only respond about math.",
            "math_tutor",
            "Specialist agent for math questions");
        ChatClientAgent triageAgent = new(chatClient.AsIChatClient(),
            "You determine which agent to use based on the user's homework question. ALWAYS handoff to another agent.",
            "triage_agent",
            "Routes messages to the appropriate specialist agent");
        var workflow = AgentWorkflowBuilder.CreateHandoffBuilderWith(triageAgent)
            .WithHandoffs(triageAgent, [mathTutor, historyTutor])
            .WithHandoffs([mathTutor, historyTutor], triageAgent)
            .Build();

        List<Microsoft.Extensions.AI.ChatMessage> messages = [];
        while (true)
        {
            string question = await Microsoft.DotNet.Interactive.Kernel.GetInputAsync("Question");
            messages.Add(new(ChatRole.User, question!));
            messages.AddRange(await RunWorkflowAsync(workflow, messages));
        }

    case "groupchat":
        IReadOnlyList<AIAgent> translationAgents = (from lang in (string[])["French", "Spanish", "English"] select GetTranslationAgent(lang, chatClient)).ToArray();
        await new SimpleRoundRobinGroupChat(translationAgents, maximumIterationCount: 5)
            .RunAsync([new(ChatRole.User, "Hello, world!")]);
        break;

    default:
        throw new InvalidOperationException("Invalid workflow type.");
}

Choose workflow type ('sequential', 'concurrent', 'handoffs', 'groupchat'): 

In [30]:
var roundRobinTestAgents = new[]
{
    GetTranslationAgent("French", chatClient),
    GetTranslationAgent("Spanish", chatClient),
    GetTranslationAgent("English", chatClient),
};

var groupChat = new SimpleRoundRobinGroupChat(roundRobinTestAgents, maximumIterationCount: 2);
var seedMessages = new List<ChatMessage> { new(ChatRole.User, "Hello, world!") };
var roundRobinHistory = await groupChat.RunAsync(seedMessages);

Console.WriteLine($"Total messages exchanged: {roundRobinHistory.Count}");
Console.WriteLine($"Last speaker role: {roundRobinHistory.Last().Role}");


9618a91a64ea4f99be1ead258ac83660
Bonjour, le monde !
cc7e35ac9f0147f7a0f53060c2e8deb8
¡Hola, mundo!
Total messages exchanged: 3
Last speaker role: assistant
